# Cluster Profiling — Land Use Segmentation

Profile and visualize the 7 land-use segments with descriptive statistics
and t-SNE projections.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

## Profiling Steps

1. Load clustered pixel data
2. Compute cluster centroids and descriptive statistics
3. Generate t-SNE 2D projection colored by cluster
4. Create radar charts for cluster profiles
5. Map clusters back to geographic coordinates

In [ ]:
# Load clustered data
df = pd.read_parquet('../data/processed/clustered_pixels.parquet')
feature_cols = [c for c in df.columns if c != 'cluster']

# Cluster profiles
profiles = df.groupby('cluster')[feature_cols].agg(['mean', 'std'])
print('Cluster centroids (mean):')
print(df.groupby('cluster')[feature_cols].mean().round(3))
print(f'\nCluster sizes: {df.cluster.value_counts().sort_index().to_dict()}')

In [ ]:
# t-SNE projection
sample = df.sample(n=min(5000, len(df)), random_state=42)
scaler = StandardScaler()
X_sample = scaler.fit_transform(sample[feature_cols].values)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embedding = tsne.fit_transform(X_sample)

plt.figure(figsize=(10, 8))
for c in sorted(sample.cluster.unique()):
    mask = sample.cluster.values == c
    plt.scatter(embedding[mask, 0], embedding[mask, 1], s=5, alpha=0.6, label=f'Segment {c}')
plt.legend(markerscale=4)
plt.title('t-SNE Projection of Land Use Segments')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('../reports/tsne_clusters.png', dpi=150)
plt.show()